# CMRAS Future-Risk IEEE Experiments

**Simulation-only research notebook.** These experiments evaluate classification against simulated or unverified risk labels and do not establish clinical two-month prediction accuracy. This notebook never creates splits, changes labels, or accesses production services.

## 1. Environment setup

Run on a standard CPU Colab runtime. Package versions are recorded in the downloadable experiment bundle.

In [ ]:
!pip -q install pandas numpy scikit-learn xgboost matplotlib joblib

import sys, os, json, hashlib, zipfile, shutil, platform, random
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, xgboost, joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support, f1_score, roc_curve, precision_recall_curve, auc, brier_score_loss)
from xgboost import XGBClassifier

SEED = 42
random.seed(SEED); np.random.seed(SEED)
CLASSES = ['Low', 'Moderate', 'High', 'Severe']
CLASS_TO_INT = {name: i for i, name in enumerate(CLASSES)}
TARGET = 'predicted_risk_next_2_months'
NUMERIC = ['age_months', 'weight_kg', 'height_cm', 'z_score']
FEATURES = {'no_current_risk': NUMERIC, 'with_current_risk': NUMERIC + ['current_risk']}
print({'python': sys.version, 'pandas': pd.__version__, 'numpy': np.__version__, 'sklearn': sklearn.__version__, 'xgboost': xgboost.__version__})

## 2. Upload frozen experiment input

Upload the ZIP described in `README_Colab_IEEE_Experiments.md`. Its internal paths must begin with `research_experiments/`.

In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1: raise RuntimeError('Upload exactly one CMRAS_Colab_Input.zip file.')
archive_name = next(iter(uploaded))
if not archive_name.lower().endswith('.zip'): raise RuntimeError('The uploaded file must be a ZIP archive.')
WORK = Path('/content/cmras_colab_input')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir()
with zipfile.ZipFile(archive_name) as zf: zf.extractall(WORK)
roots = list(WORK.rglob('research_experiments'))
if not roots: raise FileNotFoundError('ZIP does not contain a research_experiments directory.')
RESEARCH = roots[0]
print('Using:', RESEARCH)

## 3. Frozen-data integrity gate

This gate checks manifests, hashes, expected partition sizes, namespaced groups, frozen membership files, target schema, and feature leakage before fitting a model. The test file is hash-checked here but not loaded for labels until validation selection completes.

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()

metadata = json.loads((RESEARCH/'manifests/step2d_prepared_dataset_metadata.json').read_text())
split_manifest = json.loads((RESEARCH/'manifests/split_integrity.json').read_text())
step2e_dir = RESEARCH/'outputs/step2e_20261120T154822Z'
required = [step2e_dir/'experiment_config.json', step2e_dir/'validation_metrics.json', step2e_dir/'test_metrics.json', step2e_dir/'model_comparison.csv']
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing canonical evidence: ' + '; '.join(missing))
canonical_config = json.loads((step2e_dir/'experiment_config.json').read_text())
assert canonical_config['class_order'] == CLASSES and canonical_config['target'] == TARGET
assert canonical_config['features'] == FEATURES and canonical_config['seed'] == SEED
assert metadata['source_status'] == 'simulation_only_unverified_labels' and metadata['flagged_records'] == 1276
expected_rows = {'train': 13376, 'validation': 3040, 'test': 2784}
prepared = {}; all_groups = set()
for partition, count in expected_rows.items():
    item = metadata['prepared_files'][partition]; path = RESEARCH.parent/item['path'] if (RESEARCH.parent/item['path']).exists() else RESEARCH/'prepared_data'/f'simulation_{partition}.csv'
    if sha256(path) != item['sha256']: raise RuntimeError(f'{partition} SHA-256 mismatch')
    frame = pd.read_csv(path)
    if len(frame) != count or set(frame['partition']) != {partition}: raise RuntimeError(f'{partition} count/partition mismatch')
    required_columns = {'group_id','partition',*FEATURES['with_current_risk'],TARGET}
    if not required_columns.issubset(frame.columns): raise RuntimeError(f'{partition} schema missing required columns')
    if not set(frame[TARGET]).issubset(CLASSES): raise RuntimeError(f'{partition} target values invalid')
    groups = set(frame['group_id'])
    if all_groups & groups or not all(':' in value for value in groups): raise RuntimeError('Group overlap or missing dataset namespace')
    all_groups |= groups; prepared[partition] = frame
for item in split_manifest['datasets']:
    if item.get('status') == 'created_simulation_only':
        membership = RESEARCH/item['membership_file']
        if not membership.exists(): raise FileNotFoundError(f'Missing frozen membership: {membership}')
print('Integrity gate passed:', {k: len(v) for k,v in prepared.items()}, 'groups:', len(all_groups))

## 4. Dataset visualisation

Charts use uploaded data only and do not expose child/source group identifiers.

In [ ]:
PLOT_DIR = Path('/content/cmras_colab_output/figures'); PLOT_DIR.mkdir(parents=True, exist_ok=True)
def savefig(name):
    plt.tight_layout(); plt.savefig(PLOT_DIR/f'{name}.png', dpi=300, bbox_inches='tight'); plt.savefig(PLOT_DIR/f'{name}.pdf', bbox_inches='tight'); plt.show(); plt.close()
all_data = pd.concat([prepared[k] for k in ['train','validation','test']], ignore_index=True)
counts = pd.DataFrame({'partition': list(expected_rows), 'records': list(expected_rows.values())})
ax=counts.plot.bar(x='partition', y='records', legend=False, color='#1f77b4', title='Frozen partition sizes'); ax.set_ylabel('Records'); savefig('partition_sizes')
dist = pd.crosstab(all_data['partition'], all_data[TARGET]).reindex(columns=CLASSES, fill_value=0)
dist.plot.bar(title='Class distribution by frozen partition'); plt.ylabel('Records'); savefig('class_distribution_by_partition')
pd.crosstab(all_data['source_cohort'], all_data[TARGET]).reindex(columns=CLASSES, fill_value=0).plot.bar(title='Class distribution by source cohort'); plt.ylabel('Records'); savefig('class_distribution_by_cohort')
all_data[NUMERIC].describe().T
fig, axes = plt.subplots(2,2,figsize=(10,7))
for ax, col in zip(axes.ravel(), NUMERIC):
    for label in CLASSES: ax.hist(all_data.loc[all_data[TARGET]==label,col], bins=30, alpha=.45, label=label)
    ax.set_title(col); ax.set_xlabel(col); ax.set_ylabel('Records')
axes[0,0].legend(); savefig('numeric_feature_distributions')

## 5. Training-only preprocessing and candidate models

No child ID, date, visit number, target, previous-Z, delta-Z, or future field is exposed to a model. Numeric transformations and current-risk encoding are fitted only on training data.

In [ ]:
def make_preprocessor(features):
    parts=[('numeric', Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]), NUMERIC)]
    if 'current_risk' in features:
        parts.append(('current_risk', Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]), ['current_risk']))
    return ColumnTransformer(parts, remainder='drop')

def model_builders():
    return {'dummy_majority':lambda:DummyClassifier(strategy='most_frequent'), 'logistic_regression':lambda:LogisticRegression(C=1.0,max_iter=1000,solver='lbfgs',random_state=SEED), 'random_forest':lambda:RandomForestClassifier(n_estimators=300,max_depth=12,random_state=SEED,n_jobs=-1), 'xgboost':lambda:XGBClassifier(objective='multi:softprob',num_class=4,n_estimators=300,max_depth=5,learning_rate=.05,subsample=.8,colsample_bytree=.8,random_state=SEED,eval_metric='mlogloss',n_jobs=1)}
def evaluate(y_true,y_pred,prob=None):
    pr,re,f1,support=precision_recall_fscore_support(y_true,y_pred,labels=range(4),zero_division=0)
    out={'accuracy':accuracy_score(y_true,y_pred),'balanced_accuracy':balanced_accuracy_score(y_true,y_pred),'macro_precision':pr.mean(),'macro_recall':re.mean(),'macro_f1':f1_score(y_true,y_pred,average='macro',labels=range(4),zero_division=0),'weighted_f1':f1_score(y_true,y_pred,average='weighted',labels=range(4),zero_division=0),'high_recall':re[2],'severe_recall':re[3],'per_class':{CLASSES[i]:{'precision':pr[i],'recall':re[i],'f1':f1[i],'support':int(support[i])} for i in range(4)}}
    if prob is not None: out['multiclass_brier']=float(np.mean(np.sum((prob-np.eye(4)[y_true])**2,axis=1)))
    return out

train, validation = prepared['train'], prepared['validation']
y_train=train[TARGET].map(CLASS_TO_INT).to_numpy(); y_validation=validation[TARGET].map(CLASS_TO_INT).to_numpy()
fitted, validation_metrics, validation_predictions, rows = {}, {}, {}, []
for schema, features in FEATURES.items():
    for name, builder in model_builders().items():
        pipe=Pipeline([('preprocessing',make_preprocessor(features)),('model',builder())]); pipe.fit(train[features],y_train)
        pred=pipe.predict(validation[features]).astype(int); prob=pipe.predict_proba(validation[features]); key=f'{schema}__{name}'
        fitted[key]=pipe; validation_predictions[key]=(pred,prob); result=evaluate(y_validation,pred,prob); validation_metrics[key]=result
        rows.append({'schema':schema,'model':name,**{k:result[k] for k in ['accuracy','balanced_accuracy','macro_precision','macro_recall','macro_f1','weighted_f1','high_recall','severe_recall']}})
persistence=validation['current_risk'].map(CLASS_TO_INT).to_numpy(); persistence_metrics=evaluate(y_validation,persistence)
rows.append({'schema':'reference','model':'current_risk_persistence',**{k:persistence_metrics[k] for k in ['accuracy','balanced_accuracy','macro_precision','macro_recall','macro_f1','weighted_f1','high_recall','severe_recall']}})
comparison=pd.DataFrame(rows).sort_values(['macro_f1','severe_recall','high_recall','balanced_accuracy'],ascending=False); comparison

## 6. Validation comparison and ablation

Model selection uses validation macro-F1 only. Baselines are reference methods and cannot be selected as a trained model.

In [ ]:
display(comparison.style.format(precision=4))
fig,ax=plt.subplots(figsize=(10,4)); labels=[f'{x.schema}\n{x.model}' for x in comparison.itertuples()]; ax.bar(range(len(comparison)),comparison.macro_f1,color='#1f77b4'); ax.set(xticks=range(len(comparison)),xticklabels=labels,ylabel='Validation macro-F1',ylim=(0,1),title='Validation model comparison (simulated/unverified labels)'); plt.setp(ax.get_xticklabels(),rotation=35,ha='right'); savefig('validation_macro_f1_comparison')
ablation=comparison[comparison.model.isin(['logistic_regression','random_forest','xgboost'])].pivot(index='model',columns='schema',values='macro_f1').reset_index(); display(ablation.style.format(precision=4))
x=np.arange(len(ablation)); fig,ax=plt.subplots(figsize=(7,4)); ax.bar(x-.18,ablation.no_current_risk,.36,label='Without current risk'); ax.bar(x+.18,ablation.with_current_risk,.36,label='With current risk'); ax.set(xticks=x,xticklabels=ablation.model,ylabel='Validation macro-F1',ylim=(0,1),title='Current-risk ablation'); ax.legend(); savefig('current_risk_ablation')
candidates=[r for r in rows if r['model'] not in ['dummy_majority','current_risk_persistence']]
rank={'logistic_regression':0,'random_forest':1,'xgboost':2}
selected=sorted(candidates,key=lambda r:(-r['macro_f1'],-r['severe_recall'],-r['high_recall'],-r['balanced_accuracy'],rank[r['model']]))[0]
selected_key=f"{selected['schema']}__{selected['model']}"
print('Validation-selected ML candidate:', selected_key, 'macro-F1=',selected['macro_f1'])

## 7. Final frozen test evaluation

This is the first cell that uses test labels. Do not rerun model selection after viewing this result.

In [ ]:
test=prepared['test']; y_test=test[TARGET].map(CLASS_TO_INT).to_numpy(); model=fitted[selected_key]
test_pred=model.predict(test[FEATURES[selected['schema']]]).astype(int); test_prob=model.predict_proba(test[FEATURES[selected['schema']]])
test_metrics=evaluate(y_test,test_pred,test_prob); display(pd.DataFrame([test_metrics]).drop(columns=['per_class']))
print(classification_report(y_test,test_pred,labels=range(4),target_names=CLASSES,zero_division=0))
cm=confusion_matrix(y_test,test_pred,labels=range(4)); fig,ax=plt.subplots(figsize=(5,4)); image=ax.imshow(cm,cmap='Blues'); ax.set(xticks=range(4),xticklabels=CLASSES,yticks=range(4),yticklabels=CLASSES,xlabel='Predicted risk',ylabel='True simulated label',title=f'Frozen test confusion matrix: {selected_key}');
for i in range(4):
    for j in range(4): ax.text(j,i,str(cm[i,j]),ha='center',va='center')
fig.colorbar(image,ax=ax); savefig('selected_test_confusion_matrix')
per_class=pd.DataFrame(test_metrics['per_class']).T; display(per_class.style.format(precision=4))
per_class[['precision','recall','f1']].plot.bar(title='Selected-model class-wise metrics'); plt.ylim(0,1); plt.ylabel('Score'); savefig('selected_test_class_metrics')
errors=pd.DataFrame({'true_label':[CLASSES[i] for i in y_test],'predicted_label':[CLASSES[i] for i in test_pred]}); errors[errors.true_label!=errors.predicted_label].value_counts().rename('count').reset_index().plot.bar(x='true_label',y='count',title='Prediction errors by true class'); plt.ylabel('Errors'); savefig('prediction_error_distribution')

## 8. Feature importance, ROC/PR, and calibration

Feature importance is model-specific. The selected Logistic Regression uses absolute standardized coefficients; tree models use their fitted importances. ROC/PR and calibration are one-vs-rest probability diagnostics, not clinical calibration evidence.

In [ ]:
prep=model.named_steps['preprocessing']; estimator=model.named_steps['model']; names=prep.get_feature_names_out()
if hasattr(estimator,'coef_'): importance=np.mean(np.abs(estimator.coef_),axis=0); importance_label='mean absolute coefficient'
elif hasattr(estimator,'feature_importances_'): importance=estimator.feature_importances_; importance_label='feature importance'
else: importance=None
if importance is not None:
    fi=pd.DataFrame({'feature':names,'importance':importance}).sort_values('importance'); fi.plot.barh(x='feature',y='importance',legend=False,title=f'Feature importance: {selected_key}'); plt.xlabel(importance_label); savefig('selected_feature_importance')
else: print('Feature importance is not applicable to the selected estimator.')
y_bin=label_binarize(y_test,classes=range(4)); fig,axes=plt.subplots(1,2,figsize=(12,4))
for i,label in enumerate(CLASSES):
    fpr,tpr,_=roc_curve(y_bin[:,i],test_prob[:,i]); precision,recall,_=precision_recall_curve(y_bin[:,i],test_prob[:,i]); axes[0].plot(fpr,tpr,label=f'{label} AUC={auc(fpr,tpr):.3f}'); axes[1].plot(recall,precision,label=label)
axes[0].plot([0,1],[0,1],'k--'); axes[0].set(xlabel='False positive rate',ylabel='True positive rate',title='One-vs-rest ROC'); axes[1].set(xlabel='Recall',ylabel='Precision',title='One-vs-rest precision-recall'); axes[0].legend(); axes[1].legend(); savefig('selected_roc_pr_curves')
fig,ax=plt.subplots(figsize=(6,4))
for i,label in enumerate(CLASSES):
    bins=pd.qcut(test_prob[:,i],q=min(10,len(np.unique(test_prob[:,i]))),duplicates='drop'); cal=pd.DataFrame({'p':test_prob[:,i],'y':y_bin[:,i],'bin':bins}).groupby('bin',observed=True).mean(); ax.plot(cal.p,cal.y,marker='o',label=label)
ax.plot([0,1],[0,1],'k--'); ax.set(xlabel='Mean predicted probability',ylabel='Observed simulated-label frequency',title='One-vs-rest calibration diagnostic'); ax.legend(); savefig('selected_probability_calibration')

## 9. Compare with canonical Step 2E saved evidence

Differences are reported, never hidden. Current-risk persistence cannot be compared from saved predictions if its canonical prediction file is absent.

In [ ]:
saved_validation=json.loads((step2e_dir/'validation_metrics.json').read_text()); saved_test=json.loads((step2e_dir/'test_metrics.json').read_text())
differences=[]
for key,metric in validation_metrics.items():
    if key in saved_validation:
        for field in ['accuracy','balanced_accuracy','macro_precision','macro_recall','macro_f1','weighted_f1','high_recall','severe_recall']:
            differences.append({'experiment':key,'metric':field,'colab_minus_step2e':float(metric[field]-saved_validation[key][field])})
for field in ['accuracy','balanced_accuracy','macro_precision','macro_recall','macro_f1','weighted_f1','high_recall','severe_recall']:
    differences.append({'experiment':'selected_test','metric':field,'colab_minus_step2e':float(test_metrics[field]-saved_test[field])})
differences=pd.DataFrame(differences); display(differences.groupby('experiment').colab_minus_step2e.agg(['min','max']))
if not (step2e_dir/'predictions/validation_reference__current_risk_persistence.csv').exists(): print('NOT VERIFIED: canonical persistence raw predictions were not saved.')

## 10. Export reproducible Colab run

This writes a new output package only. Published tables/charts exclude raw group IDs.

In [ ]:
RUN_ID='colab_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); OUT=Path('/content/cmras_colab_output')/RUN_ID; ART=OUT/'artifacts'; TABLES=OUT/'tables';
for d in [OUT,ART,TABLES,OUT/'figures',OUT/'predictions',OUT/'reports']: d.mkdir(parents=True,exist_ok=True)
shutil.copytree(PLOT_DIR,OUT/'figures',dirs_exist_ok=True)
comparison.to_csv(TABLES/'validation_model_comparison.csv',index=False); ablation.to_csv(TABLES/'current_risk_ablation.csv',index=False); per_class.to_csv(TABLES/'selected_test_per_class_metrics.csv'); differences.to_csv(TABLES/'step2e_metric_differences.csv',index=False)
pd.DataFrame([test_metrics]).drop(columns=['per_class']).to_csv(TABLES/'selected_test_metrics.csv',index=False); pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(TABLES/'selected_test_confusion_matrix.csv')
for key,pipe in fitted.items(): joblib.dump(pipe,ART/f'{key}.joblib')
# predictions deliberately omit group_id
pd.DataFrame({'true_label':[CLASSES[i] for i in y_test],'predicted_label':[CLASSES[i] for i in test_pred]}).to_csv(OUT/'predictions/selected_test_predictions.csv',index=False)
environment={'python':sys.version,'platform':platform.platform(),'pandas':pd.__version__,'numpy':np.__version__,'scikit_learn':sklearn.__version__,'xgboost':xgboost.__version__,'seed':SEED}
(OUT/'environment.json').write_text(json.dumps(environment,indent=2)); (OUT/'experiment_config.json').write_text(json.dumps({'source_step2e_run':'step2e_20261120T154822Z','features':FEATURES,'target':TARGET,'classes':CLASSES,'selection':'validation macro-F1; ties Severe recall, High recall, balanced accuracy, Logistic Regression > Random Forest > XGBoost','simulation_only':True},indent=2))
(OUT/'dataset_hashes.json').write_text(json.dumps({k:{'sha256':metadata['prepared_files'][k]['sha256'],'rows':len(prepared[k])} for k in prepared},indent=2))
model_card='''# Model Card\n\nThis Colab run evaluates classification against simulated or unverified risk labels. It is not clinical, external, temporal, or observed two-month prediction validation. Do not use it for clinical decisions.\n'''
(OUT/'model_card.md').write_text(model_card)
summary=f'''# IEEE Experiment Summary\n\nDataset: 19,200 frozen, simulation-only longitudinal records split by dataset-namespaced child group into 13,376 train, 3,040 validation, and 2,784 test records. Models: DummyClassifier, Logistic Regression, Random Forest, XGBoost, and current-risk persistence reference. Validation selected `{selected_key}` by macro-F1 `{selected['macro_f1']:.4f}`. Final frozen-test macro-F1: `{test_metrics['macro_f1']:.4f}`.\n\nThese experiments evaluate classification against simulated or unverified risk labels and do not establish clinical two-month prediction accuracy.\n'''
(OUT/'reports/ieee_summary.md').write_text(summary)
shutil.make_archive(str(OUT),'zip',OUT); print('Download package:', str(OUT)+'.zip')
files.download(str(OUT)+'.zip')

## 11. Complete experiment record

This finalizes the export with full metrics, reports, class mapping, classification reports, and archives the completed bundle.

In [ ]:
def json_safe(value):
    if isinstance(value, dict): return {str(k):json_safe(v) for k,v in value.items()}
    if isinstance(value, (list,tuple)): return [json_safe(v) for v in value]
    if isinstance(value, np.generic): return value.item()
    return value

(OUT/'class_mapping.json').write_text(json.dumps(CLASS_TO_INT,indent=2))
(OUT/'validation_metrics.json').write_text(json.dumps(json_safe(validation_metrics),indent=2))
(OUT/'test_metrics.json').write_text(json.dumps(json_safe(test_metrics),indent=2))
(OUT/'classification_report_test.json').write_text(json.dumps(classification_report(y_test,test_pred,labels=range(4),target_names=CLASSES,output_dict=True,zero_division=0),indent=2))
for key,(pred,_) in validation_predictions.items():
    report=classification_report(y_validation,pred,labels=range(4),target_names=CLASSES,output_dict=True,zero_division=0)
    (OUT/f'classification_report_validation_{key}.json').write_text(json.dumps(report,indent=2))
(OUT/'reports/reproducibility.md').write_text('Training-only preprocessing, fixed frozen partitions, class mapping, hashes, environment versions, model artifacts, metrics, figures, and predictions are included. Any Colab-versus-Step-2E differences are in tables/step2e_metric_differences.csv.\n')
shutil.make_archive(str(OUT),'zip',OUT); print('Completed downloadable package:', str(OUT)+'.zip')
files.download(str(OUT)+'.zip')